# 12 — 새 LightGBM g3 (A Multi-Hot + 변이 서명) / LR bin 50·25 비교

담당: ③ 안주형 · 11/11. **기존 10/11번과 팀 파일은 변경하지 않습니다.**

- 11번 기준: 가중 XGB(E) + LR bin50 + 기존 g3, 1:1:1
- 후보1: 가중 XGB(E) + LR bin50 + **새 g3**, 1:1:1
- 후보2: 가중 XGB(E) + LR bin25 + **새 g3**, 1:1:1
- 새 g3: E → 핫스팟×3 → 위치 구간 → 정확한 위치 → A Multi-Hot(학습 fold 5명 이상) → 변이 서명 13열
- **공통 XGB와 새 g3는 CV/홀드아웃/전체 OOF에서 한 번씩만 학습하여 두 후보가 공유**합니다.
- CV가 기준보다 높은 후보만 고정 홀드아웃 1회 평가. 홀드아웃 하락폭 ≥ −0.02일 때 통과.
- 기본적으로 전체 OOF/Test 저장은 꺼져 있습니다. 검증 후 `RUN_FULL_CV=True`, `SAVE_ARTIFACTS=True` 설정.
- 제출 CSV는 만들지 않습니다. Test는 최종 추론에만 사용합니다.


In [19]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, save_predictions, log_result
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags

set_seed()
RUN_FULL_CV = True    # 팀장 승인 후 True: 전체 OOF/Test 생성 (기존 정상 파일 재사용)
SAVE_ARTIFACTS = True # True: 3개 모델 + 앙상블 예측 파일 저장
SAVE_SUBMISSION = False # 팀 규칙: 사용하지 않음
print('ROOT', ROOT, '| XGBoost', __import__('xgboost').__version__)
print('홀드아웃 스크립트:', manifest_path)

ROOT C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | XGBoost 3.4.1
홀드아웃 스크립트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\scripts\fixed_holdout_manifest.py


In [2]:
DEVICE='cpu'
print('XGB device:', DEVICE)

XGB device: cpu


In [3]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert len(le.classes_) == 26
# 공통 홀드아웃 먼저 읽기: CV는 개발 데이터에서만 실행합니다.
dev_idx, hold_idx = holdout_module.verify()
assert len(dev_idx) + len(hold_idx) == len(y)
assert not np.intersect1d(dev_idx, hold_idx).size
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)  # 팀 함수 변경 금지
print('train', X_raw.shape, 'test', X_test_raw.shape, 'classes', len(le.classes_))

def make_E(X):
    # 시윤님 02_features E와 동일한 피처 순서/변환
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()
    return out

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns)
assert X_E.shape[1] == 9174, f'E 피처 개수 불일치: {X_E.shape[1]}'
print('E', X_E.shape, X_test_E.shape)
X_dev_E = X_E.iloc[dev_idx].reset_index(drop=True)


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
train (6201, 4384) test (2546, 4384) classes 26
E (6201, 9174) (2546, 9174)


In [4]:
# 시윤님 03_hotspot_bins.ipynb의 E+Position-bin 함수 재현 (핫스팟 미사용)
import re
from collections import Counter
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def make_features_E_sparse(X):
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )# 피처 구성 E + Position-bin CV

from scipy.sparse import hstack

class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Fold 학습 데이터에서 선택한 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_position_LR = Pipeline([
    ("features", FeaturesEWithPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])



In [5]:
# 유전자별 Hotspot 아미노산 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}


assert sum(map(len,HOTSPOTS.values())) == 35

In [6]:
def make_hotspot_features(X):
    hotspot_data = {}

    # 일반 Hotspot
    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")

        tokens = X[gene].fillna("WT").astype(str).str.split()

        for position in positions:
            col_name = f"hotspot_{gene}_{position}"

            hotspot_data[col_name] = tokens.map(
                lambda variants: int(
                    any(v.startswith(position) for v in variants)
                )
            ).astype(np.int8)

    # NPM1 frameshift
    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")

    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"]
        .fillna("WT")
        .astype(str)
        .str.contains("fs", case=False, regex=False)
        .astype(np.int8)
    )

    return pd.DataFrame(hotspot_data, index=X.index)




In [7]:
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter

# 변이 토큰의 시작에 있는 단일 영문자 + 숫자만 인식: V600E -> V600, R132H -> R132.
# 468_469LG처럼 숫자로 시작하거나 복잡한 표기는 의도적으로 제외.
EXACT_POSITION_RE = re.compile(r'^([A-Za-z])(\d+)')

def patient_exact_positions(X):
    result = []
    for row in X.itertuples(index=False, name=None):
        found = set()
        for gene, value in zip(X.columns, row):
            if gene in ('ID', 'SUBCLASS') or pd.isna(value) or str(value).strip().upper() == 'WT':
                continue
            for token in str(value).split():
                match = EXACT_POSITION_RE.match(token)
                if match:
                    found.add(f'{gene}_{match.group(1).upper()}{match.group(2)}')
        result.append(found)
    return result

class ExactPositionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients

    def fit(self, X, y=None):
        counts = Counter(key for keys in patient_exact_positions(X) for key in keys)
        self.selected_ = sorted(k for k,v in counts.items() if v >= self.min_patients)
        self.mapping_ = {key: i for i,key in enumerate(self.selected_)}
        return self

    def transform(self, X):
        if not hasattr(self, 'mapping_'):
            raise RuntimeError('ExactPositionTransformer.fit()을 먼저 실행하세요')
        rows, cols = [], []
        for i, keys in enumerate(patient_exact_positions(X)):
            for key in keys:
                j = self.mapping_.get(key)
                if j is not None:
                    rows.append(i); cols.append(j)
        return csr_matrix((np.ones(len(rows),dtype=np.float32),(rows,cols)),
                          shape=(len(X),len(self.selected_)),dtype=np.float32)

class FeaturesEWithExactPositions(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients
    def fit(self, X, y=None):
        self.exact_ = ExactPositionTransformer(self.min_patients).fit(X)
        return self
    def transform(self, X):
        return hstack([make_features_E_sparse(X), self.exact_.transform(X)],
                      format='csr',dtype=np.float32)



In [8]:
class FeaturesEHotspot3BinExact(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3):
        self.bin_size=bin_size
        self.min_patients=min_patients
        self.hotspot_multiplier=hotspot_multiplier

    def fit(self, X, y=None):
        # 선택은 반드시 이 fit에 전달된 학습 Fold 데이터에서만 수행
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size, min_patients=self.min_patients).fit(X)
        self.exact_transformer_ = ExactPositionTransformer(
            min_patients=self.min_patients).fit(X)
        return self

    def transform(self, X):
        base = make_features_E_sparse(X)
        hotspot = csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32)
                             * self.hotspot_multiplier)
        bins = self.position_transformer_.transform(X)
        exact = self.exact_transformer_.transform(X)
        return hstack([base, hotspot, bins, exact], format='csr', dtype=np.float32)

G3_PARAMS = dict(n_estimators=800, learning_rate=0.02, num_leaves=15,
                 min_child_samples=5, colsample_bytree=0.2, subsample=0.8,
                 subsample_freq=1, class_weight='balanced', random_state=42,
                 verbosity=-1, n_jobs=-1)
lgbm_g3 = Pipeline([
    ('features', FeaturesEHotspot3BinExact(bin_size=50, min_patients=5, hotspot_multiplier=3)),
    ('model', LGBMClassifier(**G3_PARAMS))
])
_demo = pd.DataFrame({'BRAF':['V600E','V600K','WT','V600E','V600E','V600E'], 'IDH1':['WT']*6})
assert ExactPositionTransformer(5).fit(_demo).selected_ == ['BRAF_V600']
print('g3 피처 변환기 준비 완료')

g3 피처 변환기 준비 완료


## 1. 팀원 제공 함수 원본: A 피처 (02_features.ipynb)

In [9]:
# A. 유전자-변이 조합 추출 함수
def extract_mutation_pairs(X):
    pairs = []

    # 샘플별로 유전자 컬럼 확인
    for _, row in X.iterrows():
        sample_pairs = []

        for gene, value in row.items():
            # 결측치 또는 WT는 제외
            if pd.isna(value) or value == "WT":
                continue

            # 공백 기준으로 개별 변이 분리
            for mutation in str(value).split():
                sample_pairs.append(f"{gene}_{mutation}")

        pairs.append(sample_pairs)

    return pairs




In [10]:
from collections import Counter

import numpy as np
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin


class MutationMultiHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=5):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # 학습 fold에서만 변이 조합별 등장 빈도 계산
        mutation_pairs = extract_mutation_pairs(X)

        # 같은 샘플에 동일한 변이가 중복되어도 1회만 집계
        counts = Counter(
            pair
            for sample in mutation_pairs
            for pair in set(sample)
        )

        # 학습 fold에서 5개 이상 샘플에 등장한 변이만 유지
        selected = sorted(
            pair for pair, count in counts.items()
            if count >= self.min_frequency
        )

        # 선택된 변이 조합과 피처 인덱스 저장
        self.feature_names_ = selected
        self.feature_to_idx_ = {
            pair: idx for idx, pair in enumerate(selected)
        }

        return self

    def transform(self, X):
        # fit에서 선택된 피처만 사용
        mutation_pairs = extract_mutation_pairs(X)

        rows = []
        cols = []

        for row_idx, sample in enumerate(mutation_pairs):
            for pair in set(sample):
                col_idx = self.feature_to_idx_.get(pair)

                if col_idx is not None:
                    rows.append(row_idx)
                    cols.append(col_idx)

        # 존재하는 변이 조합은 1, 나머지는 0
        data = np.ones(len(rows), dtype=np.float32)

        return csr_matrix(
            (data, (rows, cols)),
            shape=(len(mutation_pairs), len(self.feature_names_)),
            dtype=np.float32
        )

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_, dtype=object)

## 2. 팀원 제공 함수 원본: 변이 서명 (05_mutation_signature.ipynb)

NCBI Standard Genetic Code, translation table 1. 이 함수는 아미노산 치환에서 **가능한** 염기 치환 유형을 추정한 것으로, 실제 환자 DNA 치환을 확정하는 정보는 아닙니다.

In [11]:
import re
import numpy as np
import pandas as pd
# 표준 유전 암호표 및 염기 치환 매핑 함수 작성

from itertools import product

# NCBI Standard Genetic Code (Translation Table 1)
BASES = "TCAG"
AA = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"

CODON_TO_AA = {
    "".join(codon): AA[i]
    for i, codon in enumerate(product(BASES, repeat=3))
}

# 아미노산별 가능한 코돈
AA_TO_CODONS = {}

for codon, amino_acid in CODON_TO_AA.items():
    AA_TO_CODONS.setdefault(amino_acid, []).append(codon)

# 염기 상보 관계
COMPLEMENT = {
    "A": "T",
    "T": "A",
    "C": "G",
    "G": "C"
}

SIGNATURE_TYPES = [
    "C>A", "C>G", "C>T",
    "T>A", "T>C", "T>G"
]


def normalize_substitution(ref, alt):
    """염기 치환을 피리미딘(C/T) 기준 6종으로 정규화."""
    if ref in "AG":
        ref = COMPLEMENT[ref]
        alt = COMPLEMENT[alt]

    return f"{ref}>{alt}"


def infer_substitution_weights(ref_aa, alt_aa):
    """
    아미노산 치환을 설명할 수 있는 단일 염기 치환을 탐색하고,
    6종 치환 유형별 비율을 반환한다.
    """
    counts = dict.fromkeys(SIGNATURE_TYPES, 0)

    for ref_codon in AA_TO_CODONS.get(ref_aa, []):
        for pos in range(3):
            for alt_base in "ACGT":
                if alt_base == ref_codon[pos]:
                    continue

                alt_codon = (
                    ref_codon[:pos]
                    + alt_base
                    + ref_codon[pos + 1:]
                )

                if CODON_TO_AA[alt_codon] == alt_aa:
                    substitution = normalize_substitution(
                        ref_codon[pos], alt_base
                    )
                    counts[substitution] += 1

    total = sum(counts.values())

    if total == 0:
        return None

    return {
        key: value / total
        for key, value in counts.items()
    }


# 계산 규칙 검증
print("V600E:", infer_substitution_weights("V", "E"))
print("R175H:", infer_substitution_weights("R", "H"))


V600E: {'C>A': 0.0, 'C>G': 0.0, 'C>T': 0.0, 'T>A': 1.0, 'T>C': 0.0, 'T>G': 0.0}
R175H: {'C>A': 0.0, 'C>G': 0.0, 'C>T': 1.0, 'T>A': 0.0, 'T>C': 0.0, 'T>G': 0.0}


In [12]:
# mutation_signature(X) 구현

from functools import lru_cache

# V600E, R175H, Q61* 등만 허용
MUTATION_PATTERN = re.compile(r"^([A-Z])(\d+)([A-Z*])$")


@lru_cache(maxsize=None)
def get_mutation_weights(mutation):
    """변이 문자열을 6종 염기 치환 가중치로 변환."""
    match = MUTATION_PATTERN.fullmatch(mutation)

    if match is None:
        return None

    ref_aa, _, alt_aa = match.groups()

    # 동의 변이 제외
    if ref_aa == alt_aa:
        return None

    weights = infer_substitution_weights(ref_aa, alt_aa)

    if weights is None:
        return None

    return tuple(weights[key] for key in SIGNATURE_TYPES)


def mutation_signature(X):
    """
    환자별 아미노산 변이에서 6종 염기 치환 서명을 추정한다.

    Parameters
    ----------
    X : pd.DataFrame
        유전자별 변이 문자열이 담긴 데이터프레임

    Returns
    -------
    pd.DataFrame
        log1p 치환 개수 6열 + 치환 비율 6열
        + log1p 전체 개수 1열 = 총 13열 (float32)
    """
    counts = np.zeros((len(X), 6), dtype=np.float64)

    for row_idx, row in enumerate(X.itertuples(index=False, name=None)):
        for cell in row:
            if pd.isna(cell) or cell == "WT":
                continue

            for mutation in str(cell).split():
                weights = get_mutation_weights(mutation)

                if weights is not None:
                    counts[row_idx] += weights

    totals = counts.sum(axis=1)

    ratios = np.divide(
        counts,
        totals[:, None],
        out=np.zeros_like(counts),
        where=totals[:, None] != 0
    )

    features = np.column_stack([
        np.log1p(counts),
        ratios,
        np.log1p(totals)
    ]).astype(np.float32)

    columns = (
        [f"sig_log_{s.replace('>', '_')}" for s in SIGNATURE_TYPES]
        + [f"sig_ratio_{s.replace('>', '_')}" for s in SIGNATURE_TYPES]
        + ["sig_log_total"]
    )

    return pd.DataFrame(
        features,
        index=X.index,
        columns=columns
    )


print("함수 정의 완료")

함수 정의 완료


In [13]:
sample=pd.DataFrame({'BRAF':['V600E','WT','V600E','WT'],'TP53':['WT','R175H','R175H','WT']})
s=mutation_signature(sample)
assert s.shape==(4,13) and s.dtypes.eq(np.float32).all()
assert np.isclose(s.loc[0,'sig_ratio_T_A'],1) and np.isclose(s.loc[1,'sig_ratio_C_T'],1)
assert np.isclose(s.loc[2,'sig_ratio_T_A'],.5) and np.isclose(s.loc[2,'sig_ratio_C_T'],.5)
assert (s.iloc[3]==0).all()
print('변이 서명 단위 테스트 통과')

변이 서명 단위 테스트 통과


## 3. 새 g3 변환기 — 학습 Fold에서만 A/위치 피처 선택

In [14]:
class FeaturesEHotspot3BinExactMultiHotSignature(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3, min_frequency=5):
        self.bin_size=bin_size
        self.min_patients=min_patients
        self.hotspot_multiplier=hotspot_multiplier
        self.min_frequency=min_frequency

    def fit(self, X, y=None):
        self.position_transformer_=PositionBinTransformer(self.bin_size,self.min_patients).fit(X)
        self.exact_transformer_=ExactPositionTransformer(self.min_patients).fit(X)
        self.pair_transformer_=MutationMultiHotEncoder(self.min_frequency).fit(X)
        return self

    def transform(self, X):
        if not hasattr(self,'pair_transformer_'):
            raise RuntimeError('fit()을 먼저 실행하세요')
        base=make_features_E_sparse(X)
        hotspot=csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32)*self.hotspot_multiplier)
        bins=self.position_transformer_.transform(X)
        exact=self.exact_transformer_.transform(X)
        pairs=self.pair_transformer_.transform(X)
        signature=csr_matrix(mutation_signature(X).to_numpy(dtype=np.float32))
        assert signature.shape==(len(X),13)
        return hstack([base,hotspot,bins,exact,pairs,signature],format='csr',dtype=np.float32)

new_g3=Pipeline([
    ('features',FeaturesEHotspot3BinExactMultiHotSignature(bin_size=50,min_patients=5,hotspot_multiplier=3,min_frequency=5)),
    ('model',LGBMClassifier(**G3_PARAMS))
])
lr50=clone(pipeline_E_position_LR)
lr25=clone(pipeline_E_position_LR).set_params(features__bin_size=25)
assert new_g3.named_steps['model'].get_params()['n_estimators']==G3_PARAMS['n_estimators']
print('새 g3 / LR50 / LR25 준비 완료')

새 g3 / LR50 / LR25 준비 완료


## 4. 동일 CPU 조건으로 11번 기준 + 후보1·2 개발 CV 비교

기준 11번의 기존 g3도 같은 노트북에서 재학습합니다. **후보별 공통 모델은 재학습하지 않습니다.**

In [15]:
K=len(le.classes_)
common=dict(n_estimators=250,max_depth=3,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,
            objective='multi:softprob',num_class=K,eval_metric='mlogloss',tree_method='hist',device='cpu',
            random_state=SEED,n_jobs=-1)
xgb=XGBClassifier(**common)
def weight_array(labels):
    labels=np.asarray(labels,dtype=int)
    counts=np.bincount(labels,minlength=K)
    if np.any(counts==0): raise ValueError('학습 fold에 누락된 클래스')
    weights=1/np.sqrt(counts[labels])
    return (weights/weights.mean()).astype(np.float32)
def weighted_fit_params(labels):return {'sample_weight':weight_array(labels)}
def checked(proba,n):
    proba=np.asarray(proba,dtype=np.float64)
    assert proba.shape==(n,K) and np.isfinite(proba).all()
    assert np.allclose(proba.sum(axis=1),1,atol=1e-4)
    return proba
def fold_scores(proba,labels,cv_folds):
    return np.array([f1_score(np.asarray(labels)[va],proba[va].argmax(axis=1),average='macro',zero_division=0)
                     for tr,va in cv_folds])

cv_xgb=run_cv(xgb,X_dev_E,y_dev,folds=folds,fit_params_fn=weighted_fit_params)
cv_lr50=run_cv(lr50,X_dev_raw,y_dev,folds=folds)
cv_lr25=run_cv(lr25,X_dev_raw,y_dev,folds=folds)
cv_old_g3=run_cv(lgbm_g3,X_dev_raw,y_dev,folds=folds)
cv_new_g3=run_cv(new_g3,X_dev_raw,y_dev,folds=folds)
px=checked(cv_xgb['oof_proba'],len(y_dev)); p50=checked(cv_lr50['oof_proba'],len(y_dev))
p25=checked(cv_lr25['oof_proba'],len(y_dev)); pg_old=checked(cv_old_g3['oof_proba'],len(y_dev))
pg_new=checked(cv_new_g3['oof_proba'],len(y_dev))
probas_dev={'baseline':checked((px+p50+pg_old)/3,len(y_dev)),
            'candidate1':checked((px+p50+pg_new)/3,len(y_dev)),
            'candidate2':checked((px+p25+pg_new)/3,len(y_dev))}
scores_dev={name:fold_scores(p,y_dev,folds) for name,p in probas_dev.items()}
base_mean=float(scores_dev['baseline'].mean())
cv_pass={name:bool(scores_dev[name].mean()>base_mean) for name in ('candidate1','candidate2')}
cv_table=pd.DataFrame([{'model':name,'cv_mean':float(sc.mean()),'cv_std':float(sc.std()),
                         'delta_vs_11':float(sc.mean()-base_mean),'pass_cv':cv_pass.get(name,None)}
                       for name,sc in scores_dev.items()])
display(cv_table)
print('CV 통과 후보:',[name for name,ok in cv_pass.items() if ok])

fold 1: Macro F1 = 0.5122
fold 2: Macro F1 = 0.4631
fold 3: Macro F1 = 0.4652
fold 4: Macro F1 = 0.4789
fold 5: Macro F1 = 0.4909
CV Macro F1 = 0.4821 (std 0.0181)
fold 1: Macro F1 = 0.5133
fold 2: Macro F1 = 0.4652
fold 3: Macro F1 = 0.4629
fold 4: Macro F1 = 0.4909
fold 5: Macro F1 = 0.4982
CV Macro F1 = 0.4861 (std 0.0194)
fold 1: Macro F1 = 0.5095
fold 2: Macro F1 = 0.4655
fold 3: Macro F1 = 0.4889
fold 4: Macro F1 = 0.4854
fold 5: Macro F1 = 0.4995
CV Macro F1 = 0.4897 (std 0.0148)
fold 1: Macro F1 = 0.5137
fold 2: Macro F1 = 0.4902
fold 3: Macro F1 = 0.4939
fold 4: Macro F1 = 0.5062
fold 5: Macro F1 = 0.5014
CV Macro F1 = 0.5011 (std 0.0084)
fold 1: Macro F1 = 0.5079
fold 2: Macro F1 = 0.5115
fold 3: Macro F1 = 0.4873
fold 4: Macro F1 = 0.4957
fold 5: Macro F1 = 0.4877
CV Macro F1 = 0.4980 (std 0.0101)


,model,cv_mean,cv_std,delta_vs_11,pass_cv
0,baseline,0.512631,0.014326,0.000000,None
1,candidate1,0.513971,0.007605,0.001340,True
2,candidate2,0.511997,0.007662,-0.000634,False


CV 통과 후보: ['candidate1']


## 5. 고정 홀드아웃 1회 — CV 통과 후보만

11번 기준 모델은 비교를 위해 같은 홀드아웃에서 계산하며, 기존 g3는 1회만 학습합니다. 후보1·2가 모두 통과하면 XGB·새 g3 예측을 공유합니다. 홀드아웃을 보고 파라미터를 다시 조정하지 마세요.

In [16]:
holdout_results={}
passed=[]
if not any(cv_pass.values()):
    print('CV 미개선: 홀드아웃 건너뜀')
else:
    assert len(dev_idx)==4960 and len(hold_idx)==1241
    raw_groups=pd.util.hash_pandas_object(X_raw.fillna('<MISSING>').astype(str),index=False).to_numpy()
    assert not set(raw_groups[dev_idx]) & set(raw_groups[hold_idx]),'개발/홀드아웃 중복 원본 확인 필요'
    X_train_h=X_raw.iloc[dev_idx]; X_hold=X_raw.iloc[hold_idx]
    y_train_h=np.asarray(y)[dev_idx]; y_hold=np.asarray(y)[hold_idx]
    m_x=clone(xgb).fit(X_E.iloc[dev_idx],y_train_h,sample_weight=weight_array(y_train_h))
    m_50=clone(lr50).fit(X_train_h,y_train_h)
    m_old=clone(lgbm_g3).fit(X_train_h,y_train_h)
    m_new=clone(new_g3).fit(X_train_h,y_train_h)
    p_x=checked(m_x.predict_proba(X_E.iloc[hold_idx]),len(y_hold))
    p_50=checked(m_50.predict_proba(X_hold),len(y_hold))
    p_old=checked(m_old.predict_proba(X_hold),len(y_hold))
    p_new=checked(m_new.predict_proba(X_hold),len(y_hold))
    p_hold_base=checked((p_x+p_50+p_old)/3,len(y_hold))
    base_hold=float(f1_score(y_hold,p_hold_base.argmax(1),average='macro',zero_division=0))
    for name in ('candidate1','candidate2'):
        if not cv_pass[name]:continue
        if name=='candidate1':p_lr=p_50
        else:
            m_25=clone(lr25).fit(X_train_h,y_train_h)
            p_lr=checked(m_25.predict_proba(X_hold),len(y_hold))
        p_ens=checked((p_x+p_lr+p_new)/3,len(y_hold))
        score=float(f1_score(y_hold,p_ens.argmax(1),average='macro',zero_division=0))
        delta=score-base_hold
        ok=delta>=-0.02
        holdout_results[name]={'holdout_f1':score,'baseline_holdout_f1':base_hold,'delta':delta,'pass':ok}
        if ok:passed.append(name)
    display(pd.DataFrame.from_dict(holdout_results,orient='index'))
    print('홀드아웃 통과 후보:',passed)

,holdout_f1,baseline_holdout_f1,delta,pass
candidate1,0.518364,0.521656,-0.003292,True


홀드아웃 통과 후보: ['candidate1']


## 6. 전체 Train 5-Fold OOF / 전체 Train 재학습 Test — 통과 후보만

`RUN_FULL_CV=True`로 변경한 후 실행. **두 후보가 모두 통과하면 둘 다 저장**하며, XGB·새 g3는 공통 재사용. Test는 최종 모델의 추론에만 사용합니다.

In [20]:
PRED=ROOT/'experiments'/'preds'
if not RUN_FULL_CV:
    print('전체 OOF/Test 비활성화 — 검증 결과 확인 후 RUN_FULL_CV=True')
elif not passed:
    print('통과 후보 없음: 전체 예측 생성하지 않음')
else:
    full_folds=list(get_folds(y,X=X_raw,classes=le.classes_))
    seen=np.zeros(len(y),dtype=int)
    for tr,va in full_folds:seen[va]+=1
    assert len(full_folds)==5 and np.all(seen==1)
    def full_model(model,model_name,weighted=False,raw=False):
        x_train=X_raw if raw else X_E
        x_test=X_test_raw if raw else X_test_E
        oof=np.zeros((len(y),K),dtype=np.float64)
        for i,(tr,va) in enumerate(full_folds,1):
            m=clone(model)
            fit_kw={'sample_weight':weight_array(np.asarray(y)[tr])} if weighted else {}
            m.fit(x_train.iloc[tr],np.asarray(y)[tr],**fit_kw)
            assert np.array_equal(m.classes_,np.arange(K))
            oof[va]=checked(m.predict_proba(x_train.iloc[va]),len(va))
            print(model_name,'fold',i,'/5')
        m=clone(model)
        fit_kw={'sample_weight':weight_array(y)} if weighted else {}
        m.fit(x_train,y,**fit_kw)
        assert np.array_equal(m.classes_,np.arange(K))
        test_p=checked(m.predict_proba(x_test),len(x_test))
        return checked(oof,len(y)),test_p
    full_outputs={}
    full_outputs['xgb']=full_model(xgb,'weighted_XGB_E',weighted=True)
    full_outputs['new_g3']=full_model(new_g3,'new_g3',raw=True)
    if 'candidate1' in passed:full_outputs['lr50']=full_model(lr50,'LR_bin50',raw=True)
    if 'candidate2' in passed:full_outputs['lr25']=full_model(lr25,'LR_bin25',raw=True)
    for name in passed:
        lr_name='lr50' if name=='candidate1' else 'lr25'
        oof=checked(sum(full_outputs[k][0] for k in ('xgb',lr_name,'new_g3'))/3,len(y))
        test_p=checked(sum(full_outputs[k][1] for k in ('xgb',lr_name,'new_g3'))/3,len(X_test_raw))
        full_outputs[name]=(oof,test_p)
        print(name,'full OOF Macro F1:',f1_score(y,oof.argmax(1),average='macro',zero_division=0))

weighted_XGB_E fold 1 /5
weighted_XGB_E fold 2 /5
weighted_XGB_E fold 3 /5
weighted_XGB_E fold 4 /5
weighted_XGB_E fold 5 /5
new_g3 fold 1 /5
new_g3 fold 2 /5
new_g3 fold 3 /5
new_g3 fold 4 /5
new_g3 fold 5 /5
LR_bin50 fold 1 /5
LR_bin50 fold 2 /5
LR_bin50 fold 3 /5
LR_bin50 fold 4 /5
LR_bin50 fold 5 /5
candidate1 full OOF Macro F1: 0.5323652674584385


## 7. 예측 저장 — 통과 후보별 3종 파일 (OOF / Test / Meta)

기존 예측 파일을 덮어쓰지 않습니다. PR에는 이 노트북과 새 예측 산출물만 포함하세요. 제출 CSV는 생성하지 않습니다.

In [21]:
import os,tempfile
if not SAVE_ARTIFACTS:
    print('저장 비활성화 — SAVE_ARTIFACTS=True 설정 후 이 셀 실행')
elif not RUN_FULL_CV or not passed or 'full_outputs' not in globals():
    print('전체 OOF/Test 및 통과 후보 필요')
else:
    PRED.mkdir(parents=True,exist_ok=True)
    stems={
        'xgb':'A_model__12_weighted_xgb_E_cpu',
        'new_g3':'A_model__12_g3_multihot_signature',
        'lr50':'A_model__12_lr_position_bin50',
        'lr25':'A_model__12_lr_position_bin25',
        'candidate1':'A_model__12_ensemble_g3_signature_lr50',
        'candidate2':'A_model__12_ensemble_g3_signature_lr25',
    }
    for name,(oof,test_p) in full_outputs.items():
        stem=stems[name]
        meta={'model':name,'classes':list(map(str,le.classes_)),
              'train_rows':len(y),'test_rows':len(X_test_raw),'seed':int(SEED),'folds':5,
              'test_fit':'full_train','experiment':'12_g3_multihot_signature',
              'g3_params':G3_PARAMS if name=='new_g3' else None,
              'cv_mean':float(scores_dev[name].mean()) if name in scores_dev else None,
              'holdout':holdout_results.get(name),
              'sources':[stems['xgb'],stems['lr50' if name=='candidate1' else 'lr25'],stems['new_g3']] if name.startswith('candidate') else None}
        for kind,arr in [('oof',oof),('test',test_p)]:
            path=PRED/f'{stem}__{kind}.npy'
            arr=np.asarray(checked(arr,len(arr)),dtype=np.float32)
            if path.exists():
                old=np.load(path,allow_pickle=False)
                if old.shape!=arr.shape or not np.allclose(old,arr,atol=1e-5):
                    raise ValueError('기존 파일 내용 다름: '+str(path))
                print('기존 파일 유지:',path.name)
            else:
                fd,tmp=tempfile.mkstemp(dir=PRED,suffix='.npy');os.close(fd)
                try:np.save(tmp,arr);os.replace(tmp,path)
                finally:
                    if os.path.exists(tmp):os.unlink(tmp)
                print('저장:',path.name)
        meta_path=PRED/f'{stem}__meta.json'
        if meta_path.exists():
            old=json.loads(meta_path.read_text(encoding='utf-8'))
            if old!=meta:raise ValueError('기존 메타 불일치: '+str(meta_path))
        else:meta_path.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
    print('저장 완료. 제출 CSV는 생성하지 않았습니다.')

저장: A_model__12_weighted_xgb_E_cpu__oof.npy
저장: A_model__12_weighted_xgb_E_cpu__test.npy
저장: A_model__12_g3_multihot_signature__oof.npy
저장: A_model__12_g3_multihot_signature__test.npy
저장: A_model__12_lr_position_bin50__oof.npy
저장: A_model__12_lr_position_bin50__test.npy
저장: A_model__12_ensemble_g3_signature_lr50__oof.npy
저장: A_model__12_ensemble_g3_signature_lr50__test.npy
저장 완료. 제출 CSV는 생성하지 않았습니다.


## 8. PR 체크

- [ ] 기준 11번·후보1·후보2 **같은 CPU·같은 개발 Fold** 비교
- [ ] A 피처의 `min_frequency=5`는 각 학습 Fold에서만 fit
- [ ] 변이 서명은 행별 계산·13열·float32
- [ ] CV 개선 후보만 홀드아웃 1회, 하락폭 ≥ −0.02
- [ ] 통과 후보만 전체 6,201 OOF / 2,546 Test 생성
- [ ] 후보 둘 다 통과하면 둘 다 예측 저장, **PR 하나**
- [ ] 기존 06/10/11번 및 팀원 파일 수정하지 않음
- [ ] 제출 CSV 생성하지 않음

**주의:** 현재 코드는 업로드된 노트북 기준으로 통합·정적 검사한 버전입니다. 실제 프로젝트 데이터와 `src` 환경에서 전체 학습은 아직 실행하지 않았습니다.